In [39]:
## Import libraries (numpy, pandas, ...)
import pandas as pd
import numpy as np
from typing import Dict
import statsmodels.api as sm

In [40]:
## Import Divvy trip data
divvy_trips = pd.read_stata("data/divvy_data.dta")
divvy_trips

,start_date,from_station_id,trips
0,2013-06-27,17,4.0
1,2013-06-27,19,2.0
2,2013-06-27,20,1.0
3,2013-06-27,24,1.0
4,2013-06-27,28,1.0
...,...,...,...
951667,2019-12-31,659,3.0
951668,2019-12-31,660,2.0
951669,2019-12-31,664,1.0
951670,2019-12-31,672,21.0


In [41]:
## Import Divvy location dataset
divvy_locations = pd.read_stata("data/IDlatlong.dta")
divvy_locations

,from_station_id,Latitude,Longitude
0,2.0,41.876511,-87.620548
1,3.0,41.867226,-87.615355
2,4.0,41.856268,-87.613348
3,5.0,41.874053,-87.627716
4,6.0,41.886976,-87.612813
...,...,...,...
604,664.0,41.939354,-87.683282
605,665.0,41.747363,-87.580046
606,666.0,41.907221,-87.655618
607,672.0,41.891023,-87.635480


In [42]:
## Merge Datasets
divvy = pd.merge(divvy_trips, divvy_locations, left_on="from_station_id", right_on="from_station_id", how="left")
divvy

,start_date,from_station_id,trips,Latitude,Longitude
0,2013-06-27,17,4.0,41.903119,-87.673935
1,2013-06-27,19,2.0,41.868968,-87.659141
2,2013-06-27,20,1.0,41.910522,-87.653106
3,2013-06-27,24,1.0,41.891847,-87.620580
4,2013-06-27,28,1.0,41.914680,-87.643320
...,...,...,...,...,...
951667,2019-12-31,659,3.0,41.895501,-87.682017
951668,2019-12-31,660,2.0,42.004583,-87.661406
951669,2019-12-31,664,1.0,41.939354,-87.683282
951670,2019-12-31,672,21.0,41.891023,-87.635480


In [43]:
# Haversine distance in miles
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c



def div_filter( divvy : pd.DataFrame, c : tuple[float], r :float):
###
# dataset filtering function; takes center, radius info for filtering
# params: 
#   divvy (DataFrame): DataFrame w/ divvy bike ride info including longitude and latitude values for station, station ID, and date (YYY-MM-DD)
#   c (tuple(float)) : (c is short for "center"), this is the center longitude and latitude tuple defining the midpoint of the area for which the divvy data will be filtered for distance 
#   r (float) : radius (in Mi), the radius defining the circular boundary around the center ('c') point (divvy ride entried within this radius will be added to the output dataframe)
# outputs: 
#   inrange (DataFrame) : a dataframe filtered for divvy bikle rides within the specified radius around the provided center point
###
  # coordinates lat, long sep into single vars for computations
  lat = c[0]
  lon = c[1]

  # Calculate distance from each station to Soldier Field
  divvy["dist_to_c_mi"] = haversine_miles(
      lat,
      lon,
      divvy["Latitude"],
      divvy["Longitude"]
  )

  inrange = divvy[divvy["dist_to_c_mi"] <= r].copy()
  return inrange
  

In [52]:
def split_df_on_game_days(divvy: pd.DataFrame, gds_by_yr: dict):
    all_gds = set()
    for dates in gds_by_yr.values():
        all_gds |= dates

    # Build a mask: within any season's first-to-last game range
    in_season = pd.Series(False, index=divvy.index)
    for yr_dates in gds_by_yr.values():
        season_start = min(yr_dates)
        season_end   = max(yr_dates)
        in_season |= (divvy["start_date"] >= season_start) & (divvy["start_date"] <= season_end)

    game_day_mask = divvy["start_date"].isin(all_gds)

    gd    = divvy[game_day_mask].copy(); print("GAME DAY"); print(gd)
    notgd = divvy[in_season & ~game_day_mask].copy(); print("\033[1m" +"*NOT* " + "\033[0m" + "GAME DAY"); print(notgd)

    return gd, notgd


In [53]:
soldiers_coords = (41.8625, -87.6167)
# returns df of all divvy ride entries within 1 mi of soldiers field center pt as defined above
soldiers_df = div_filter(divvy, soldiers_coords, r=1.0) 

jackson_coords = (41.7831, -87.5819)
jackson_df = div_filter(divvy, jackson_coords, r=1.0) # does the same but now for jackson park (control area)



#now need to create a dict to hold the game days, will use the years as keys then use a set to hold the dates in same format as the divvy df does ('YYY-MM-DD' string)
gds_by_yr = {
    '2017': {
        pd.Timestamp('2017-09-10'),
        pd.Timestamp('2017-09-24'),
        pd.Timestamp('2017-10-09'),
        pd.Timestamp('2017-10-22'),
        pd.Timestamp('2017-11-12'),
        pd.Timestamp('2017-11-19'),
        pd.Timestamp('2017-12-03'),
    },
    '2018': {
        pd.Timestamp('2018-09-17'),
        pd.Timestamp('2018-09-30'),
        pd.Timestamp('2018-10-21'),
        pd.Timestamp('2018-10-28'),
        pd.Timestamp('2018-11-11'),
        pd.Timestamp('2018-11-18'),
        pd.Timestamp('2018-12-09'),
        pd.Timestamp('2018-12-16'),
    }
}

# now need to separate based on game-day or non-game day entries
s_gd, s_notgd = split_df_on_game_days(soldiers_df, gds_by_yr)
j_gd, j_notgd = split_df_on_game_days(jackson_df, gds_by_yr)


GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
553997 2017-09-10                2  101.0  41.876511 -87.620548      0.988131
553998 2017-09-10                3  247.0  41.867226 -87.615355      0.333785
553999 2017-09-10                4  130.0  41.856268 -87.613348      0.463860
554000 2017-09-10                5   23.0  41.874053 -87.627716      0.979012
554034 2017-09-10               41   37.0  41.872078 -87.629544      0.935233
...           ...              ...    ...        ...        ...           ...
768771 2018-12-16              341   18.0  41.866095 -87.607267      0.545252
768786 2018-12-16              370    2.0  41.854184 -87.619154      0.588281
768796 2018-12-16              394    7.0  41.870816 -87.631246      0.943576
768894 2018-12-16              623   16.0  41.872773 -87.623981      0.802603
768897 2018-12-16              626   11.0  41.867491 -87.632190      0.868451

[313 rows x 6 columns]
*NOT* GAME DAY
       start_dat

In [49]:
#now with the 4 separate datasets (for A,B,C,D analagously in DiD table),
# can do the DiD calcualtion using th statsmodels api....

s_gd["treated"] = 1; s_gd["game_day"] = 1
s_notgd["treated"] = 1; s_notgd["game_day"] = 0
j_gd["treated"] = 0; j_gd["game_day"] = 1
j_notgd["treated"] = 0; j_notgd["game_day"] = 0

combined = pd.concat ([s_gd, s_notgd, j_gd, j_notgd], ignore_index=True)
daily = (combined.groupby(["start_date", "treated", "game_day"])["trips"].sum().reset_index())

daily["DiD"] = daily["treated"] * daily["game_day"]

#now can run OLS
X = sm.add_constant(daily[["treated", "game_day", "DiD"]])
y = daily["trips"]

model = sm.OLS(y, X).fit()
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:                  trips   R-squared:                       0.485
Model:                            OLS   Adj. R-squared:                  0.480
Method:                 Least Squares   F-statistic:                     109.1
Date:                Mon, 18 May 2026   Prob (F-statistic):           8.13e-50
Time:                        21:45:32   Log-Likelihood:                -2430.3
No. Observations:                 352   AIC:                             4869.
Df Residuals:                     348   BIC:                             4884.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         77.8944     19.110      4.076      0.0